In [2]:
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

# ==========================================
# 1. PENGATURAN PATH & CONFIGURATION
# ==========================================
# Pastikan script python ini berada di folder yang sama dengan folder 'data'
DATA_DIR = "bukan fruit" 
IMG_SIZE = (224, 224) # Mengubah semua ukuran gambar menjadi 224x224 piksel
BATCH_SIZE = 32

# ==========================================
# 2. LOADING DATASET (Otomatis Split Train & Val)
# ==========================================
# Memuat data untuk Training (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Memuat data untuk Validasi/Testing (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Mengambil nama-nama kelas berdasarkan nama folder
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nTotal Kelas ditemukan: {num_classes}")
print(f"Daftar Kelas: {class_names}\n")

# Optimasi performa loading data ke memori
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# ==========================================
# 3. MEMBANGUN MODEL CNN (Convolutional Neural Network)
# ==========================================
model = models.Sequential([
    # Rescaling untuk mengubah nilai pixel dari 0-255 menjadi 0-1
    layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    
    # Feature Extraction (Mempelajari pola gambar)
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Flattening & Fully Connected Layer (Klasifikasi keputusan)
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Mencegah overfitting
    layers.Dense(num_classes, activation='softmax') # Output sesuai jumlah kelas
])

# ==========================================
# 4. COMPILE & TRAINING MODEL
# ==========================================
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

# Mulai proses training (silakan naikkan epochs jika akurasi kurang tinggi)
EPOCHS = 50
print("\nMemulai Training Model...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# ==========================================
# 5. MENYIMPAN MODEL
# ==========================================
model.save('model_klasifikasi_makanan.h5')
print("\nModel sukses dilatih dan disimpan dengan nama 'model_klasifikasi_makanan.h5'!")

Found 1784 files belonging to 6 classes.
Using 1428 files for training.
Found 1784 files belonging to 6 classes.
Using 356 files for validation.

Total Kelas ditemukan: 6
Daftar Kelas: ['Ayam Goreng', 'Ikan Goreng', 'Nasi Goreng', 'ayam_goreng', 'telur_balado', 'telur_dadar']



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,734 (42.61 MB)

 Trainable params: 11,169,734 (42.61 MB)

 Non-trainable params: 0 (0.00 B)


Memulai Training Model...
Epoch 1/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 25s 508ms/step - accuracy: 0.3235 - loss: 1.7347 - val_accuracy: 0.4579 - val_loss: 1.3269
Epoch 2/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.4608 - loss: 1.3358 - val_accuracy: 0.4691 - val_loss: 1.2745
Epoch 3/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - accuracy: 0.4811 - loss: 1.2675 - val_accuracy: 0.5646 - val_loss: 1.1190
Epoch 4/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.5749 - loss: 1.1209 - val_accuracy: 0.5169 - val_loss: 1.1685
Epoch 5/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - accuracy: 0.6134 - loss: 1.0196 - val_accuracy: 0.6067 - val_loss: 1.0564
Epoch 6/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.6611 - loss: 0.8789 - val_accuracy: 0.6124 - val_loss: 1.0029
Epoch 7/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.7353 - loss: 0.7103 - val_accuracy: 0.6517 - val_loss: 0.9694
Epoch 8/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.7892 - loss: 0.5964 - v


Model sukses dilatih dan disimpan dengan nama 'model_klasifikasi_makanan.h5'!


In [3]:
import numpy as np
import tensorflow as tf

# Load kembali model yang sudah dilatih
model = tf.keras.models.load_model('model_klasifikasi_makanan.h5')

# Daftar kelas harus urut sesuai abjad (sama seperti saat training)
class_names = ['Ayam_goreng', 'ayam_goreng', 
               'Ikan Goreng',  'Nasi Goreng',
               'telur_balado', 'telur_dadar', ]

# Ganti dengan path foto makanan/buah yang mau kamu tes
img_path = 'D:/Rixh/bukan fruit/telur_balado/telur_balado (7).jpg' 

# Load gambar dan sesuaikan ukurannya
img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Buat batch axis

# Prediksi gambar
predictions = model.predict(img_array)
score = tf.nn.softmax(predictions[0])

# Tampilkan hasil
print(f"Gambar ini kemungkinan besar adalah: {class_names[np.argmax(score)]} "
      f"dengan tingkat keyakinan {100 * np.max(score):.2f}%.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
Gambar ini kemungkinan besar adalah: telur_balado dengan tingkat keyakinan 35.22%.
